In [1]:
#Required Packages

library(extraDistr)
library(fdrtool)

#Pipeline for the Simulation under Different Configurations and Number of Users

run_pipeline <- function(config, N) {

#Unconstrained Estimated Preference Function

hat_h <- function(W, NN, x_points) {
  w <- W$Freq
  y_sum <- NN
  x_cum <- cumsum(w)
  y_cum <- cumsum(y_sum)
  x <- c(0, x_cum)
  y <- c(0, y_cum)
  gg <- gcmlcm(x, y)
  assign_slope <- function(x_point) {
    for (i in 1:(length(gg$x.knots) - 1)) {
      if (x_point > gg$x.knots[i] &&
          x_point <= gg$x.knots[i + 1]) {
        return(gg$slope.knots[i])
      }
    }
    return(NA)
  }
  slopes <- sapply(x_cum, assign_slope)
  X_vals <- W$X_raw
  sapply(x_points, function(x_point) {
    idx <- max(which(X_vals <= x_point))
    if (is.finite(idx)) slopes[idx] else NA
  })
}

#Constrained Estimated Preference Function

hat_h0 <- function(theta, W, NN, r0, x_points) {
  w <- W$Freq
  y_sum <- NN
  x_cum <- cumsum(w)
  y_cum <- cumsum(y_sum)
  X_vals <- W$X_raw
  s0 <- max(which(X_vals <= r0))
  x_left <- c(0, x_cum[1:s0])
  y_left <- c(0, y_cum[1:s0])
  gcm_left <- gcmlcm(x_left, y_left)
  assign_left <- function(x_point) {
    for (i in 1:(length(gcm_left$x.knots)-1)) {
      if (x_point > gcm_left$x.knots[i] &&
          x_point <= gcm_left$x.knots[i+1]) {
        return(gcm_left$slope.knots[i])
      }
    }
    NA
  }
  slopes_left <- sapply(x_cum[1:s0], assign_left)
  slopes_left <- pmin(slopes_left, theta)
  if (s0 < length(x_cum)) {
    x_right <- x_cum[(s0+1):length(x_cum)] - x_cum[s0]
    y_right <- y_cum[(s0+1):length(y_cum)] - y_cum[s0]
    x_right <- c(0, x_right)
    y_right <- c(0, y_right)
    gcm_right <- gcmlcm(x_right, y_right)
    assign_right <- function(x_point) {
      for (i in 1:(length(gcm_right$x.knots)-1)) {
        if (x_point > gcm_right$x.knots[i] &&
            x_point <= gcm_right$x.knots[i+1]) {
          return(gcm_right$slope.knots[i])
        }
      }
      NA
    }
    slopes_right <- sapply(x_right[-1], assign_right)
    slopes_right <- pmax(slopes_right, theta)
   } else {
    slopes_right <- numeric(0)
  }
  slopes_all <- c(slopes_left, slopes_right)
  sapply(x_points, function(x_point) {
    idx <- max(which(X_vals <= x_point))
    if (is.finite(idx)) slopes_all[idx] else NA
  })
}

#Likelihood Function
    
loglik_from_slopes <- function(slopes, W, NN) {
   w <- W$Freq
   y <- NN
   eps <- 1e-12
   h <- pmin(pmax(slopes, eps), 1 - eps)
   sum(y * log(h) + (w - y) * log(1 - h))
  }  

#Kernel Specifications used in the Oracle Estimate

kernel_est <- function(r, R, U, h) {
    w <- dnorm((R - r) / h)
    sum(w * U) / sum(w)
  }                    
bandwidth <- 0.01

#A Function for Obtaining Root 

compute_root <- function(f, lower, upper) {
    f_low  <- f(lower)
    f_high <- f(upper)
    if (is.na(f_low) || is.na(f_high)) return(NA)
    if (f_low * f_high > 0) return(NA)   
    uniroot(f, lower = lower, upper = upper)$root
}

#Grid for Evaluating Test Error

grid_r <- seq(0, 1, length.out = 100)

#Confidence Interval(s) for h(pot_grid)

pot_grid  <- c(1/3)

#Parameters

n <- 20                                                                              #Maximum Number of Choices Made by 
                                                                                      #Each User                                          
mu <- 20                                                                             #Mean of the Poisson Distributed 
                                                                                      #Number of Choices Made by Each User
T <- 100                                                                             #Number of Replications
crit <- 2.27                                                                         #0.95th Quantile of the Distribution D

#Preference Function

p <- 0.7
h <- function(x) {
  (2*p - 1) * x^2 + 1 - p  
}

#Initialization

gg_slopes_list <- vector("list", T)
W_X_list       <- vector("list", T)
class_err      <- numeric(T)
mean_tr_err    <- numeric(T)
mean_err       <- numeric(T)
mean_ece       <- numeric(T)
test_points_list     <- vector("list", T)
gg_true_slopes_list  <- vector("list", T)
gg_hat_slopes_list   <- vector("list", T)
reliability_list     <- vector("list", T)
CIL <- matrix(NA, nrow = length(pot_grid), ncol = T)
CIR <- matrix(NA, nrow = length(pot_grid), ncol = T)

#Replicated Experiments

set.seed(2025)

for (t in 1:T) {

#Data Generation

  R_list <- vector("list", N)
  U_list <- vector("list", N)
  for (j in 1:N) {
    if (config %in% c(1,3,5)) {                                                     #Constant Number of Choices
         L_j <- n
      } else {                                                                      #Random Number of Choices
         L_j <- max(min(rpois(1, mu), n), 4)
    }                                                                      
    if(config %in% c(1,2)) {
       I_j = rep(1, L_j+1)                                                          #Equal Intensity                                             
       R_j <- numeric(L_j+1)
       U_j <- numeric(L_j+1)
       temp <- runif(1)
       U_j[1] <- ifelse(temp <= 0.5, 1, 0)
       R_j[1] <- U_j[1]
       if (L_j > 0) {
         for (k in 2:(L_j+1)) {
           temp1 <- runif(1)
           U_j[k] <- ifelse(temp1 <= (2*p - 1)*(R_j[k-1])^2 + 1 - p, 1, 0)
           R_j[k] <- ((sum(I_j[1:(k-1)]))*R_j[k-1] + I_j[k]*U_j[k])/(sum(I_j[1:k]))
         }
       } 
    }
    if(config %in% c(3,4)) {
       I_j = runif(L_j, 0,1)                                                        #Uniform Intensity                                                 
       R_j <- numeric(L_j+1)
       U_j <- numeric(L_j+1)
       temp <- runif(1)
       U_j[1] <- ifelse(temp <= 0.5, 1, 0)
       R_j[1] <- U_j[1]
       if (L_j > 0) {
         for (k in 2:(L_j+1)) {
           temp1 <- runif(1)
           U_j[k] <- ifelse(temp1 <= (2*p - 1)*(R_j[k-1])^2 + 1 - p, 1, 0)
           R_j[k] <- ((sum(I_j[1:(k-1)]))*R_j[k-1] + I_j[k]*U_j[k])/(sum(I_j[1:k]))
         }
       } 
    }
    else {
       I_j = numeric(L_j+1)
       R_j <- numeric(L_j+1)
       U_j <- numeric(L_j+1)
       temp <- runif(1)
       U_j[1] <- ifelse(temp <= 0.5, 1, 0)
       I_j[1] = runif(1)
       R_j[1] <- U_j[1]
       if (L_j > 0) {
         for (k in 2:(L_j+1)) {
           temp1 <- runif(1)
           U_j[k] <- ifelse(temp1 <= (2*p - 1)*(R_j[k-1])^2 + 1 - p, 1, 0)
           temp2 <- runif(1)
           I_j[k] <- ifelse(temp2 <= 0.2, I_j[k-1], runif(1))                       #Piecewise Uniform Intensity
           R_j[k] <- ((sum(I_j[1:(k-1)]))*R_j[k-1] + I_j[k]*U_j[k])/(sum(I_j[1:k]))
         }  
       }
   }   
    R_list[[j]] <- R_j
    U_list[[j]] <- U_j
  }
  

  #Training Data
    
  X_raw <- unlist(lapply(R_list, function(rj) {
    Lj <- length(rj) - 1
    Tj <- floor(Lj / 2)
    if (Tj >= 2) rj[1:Tj] else numeric(0)
  }))
  Y_raw <- unlist(lapply(U_list, function(uj) {
    Lj <- length(uj) - 1
    Tj <- floor(Lj / 2)
    if (Tj >= 2) uj[2:(Tj + 1)] else numeric(0)
  }))
  W <- as.data.frame(table(X_raw))
  W$X_raw <- as.numeric(as.character(W$X_raw))
  NN <- as.vector(rowsum(Y_raw, X_raw)[as.character(W$X_raw), ])

  #Test Data

  R_test <- unlist(lapply(R_list, function(rj) {
    Lj <- length(rj) - 1
    Tj <- floor(Lj / 2)
    rj[(Tj + 1):Lj]
  }))
  U_test <- unlist(lapply(U_list, function(uj) {
    Lj <- length(uj) - 1
    Tj <- floor(Lj / 2)
    uj[(Tj + 2):(Lj + 1)]
  }))

  #Test Error
  
  gg_hat_slopes <- hat_h(W, NN, grid_r)
  gg_true_slopes <- sapply(grid_r, kernel_est, R = R_test, U = U_test, h = bandwidth)
  mean_err[t] <- mean(abs(gg_hat_slopes - gg_true_slopes), na.rm = TRUE)
  gg_true_slopes_list[[t]] <- gg_true_slopes
  gg_hat_slopes_list[[t]] <- gg_hat_slopes

  #Empirical Calibration Error
    
  B <- 50
  pred_probs <- hat_h(W, NN, R_test)
  true_labels <- U_test              
  bin_ids <- cut(pred_probs,
                 breaks = seq(0, 1, length.out = B + 1),
                 include.lowest = TRUE, right = FALSE)
  reliability_data <- aggregate(
    cbind(pred_probs, true_labels),
    by = list(bin = bin_ids),
    FUN = mean
  )
  reliability_list[[t]] <- reliability_data
  ece <- 0
  for (b in levels(bin_ids)) {
    idx <- which(bin_ids == b)
    if (length(idx) > 0) {
      acc_b  <- mean(true_labels[idx])       
      conf_b <- mean(pred_probs[idx])      
      ece <- ece + (length(idx)/length(pred_probs)) * abs(acc_b - conf_b)
    }
  }
  mean_ece[t] <- ece  

  #Unconstrained Likelihood 
    
  p_hat <- hat_h(W, NN, W$X_raw)
  ll1 <- loglik_from_slopes(p_hat, W, NN)

  #Constrained Likelihood and the Likelihood Ratio Statistic
    
  for (pi in seq_along(pot_grid)) {
   r0 <- pot_grid[pi]
   theta_hat <- hat_h(W, NN, r0)[1]
   compute_LRT <- function(theta) {
     p_hat0 <- hat_h0(theta, W, NN, r0, W$X_raw)
     ll0 <- loglik_from_slopes(p_hat0, W, NN)
     2 * (ll1 - ll0)
   }
   f_root <- function(th) compute_LRT(th) - crit
   lower <- 0
   upper <- 1
   if (f_root(lower) * f_root(theta_hat) <= 0) {
     CIL[pi, t] <- compute_root(f_root, lower, theta_hat)
   } else {
     CIL[pi, t] <- NA
   }
   if (f_root(theta_hat) * f_root(upper) <= 0) {
     CIR[pi, t] <- compute_root(f_root, theta_hat, upper)
   } else {
     CIR[pi, t] <- NA
   }
 }  
}

#True Value of h(pot_grid)

true_vals <- h(pot_grid)

#Average Length and Coverage Probability of Confidence Interval(s)

coverage <- numeric(length(pot_grid))
avg_length <- numeric(length(pot_grid))
valid_intervals <- numeric(length(pot_grid))

for (pi in seq_along(pot_grid)) {
  cil <- CIL[pi, ]
  cir <- CIR[pi, ]
  coverage[pi] <- mean(true_vals[pi] >= cil & true_vals[pi] <= cir, na.rm = TRUE)
  avg_length[pi] <- mean(cir - cil, na.rm = TRUE)
  valid_intervals[pi] <- mean(cir > cil, na.rm = TRUE)
}           
    
cat("Result for Configuration", config, "with Number of Users", N, "\n")
cat("----------------------------------------------------", "\n")
    
#Table 1
    
cat("Mean Test Error:", round(mean(mean_err), 3), "\n")  
cat("SD Test Error:", round(sd(mean_err), 3), "\n")  
cat("Mean ECE:", round(mean(mean_ece), 3), "\n")   
cat("SD ECE:", round(sd(mean_ece), 3), "\n")   

#Table 2

cat("Coverage:", round(coverage, 3), "\n")      
cat("Length:", round(avg_length, 3), "\n")   
    
cat("End of Result", "\n")
cat("-------------", "\n")
}

configs <- c(1:6)
N_vals  <- c(300,600,900,1200,1500)

results <- do.call(rbind,
  lapply(configs, function(c) {
    do.call(rbind,
      lapply(N_vals, function(n) {
        run_pipeline(c, n)
      })
    )
  })
)

Result for Configuration 1 with Number of Users 300 
---------------------------------------------------- 
Mean Test Error: 0.052 
SD Test Error: 0.007 
Mean ECE: 0.027 
SD ECE: 0.008 
Coverage: 0.94 
Length: 0.08 
End of Result 
------------- 
Result for Configuration 1 with Number of Users 600 
---------------------------------------------------- 
Mean Test Error: 0.037 
SD Test Error: 0.004 
Mean ECE: 0.021 
SD ECE: 0.006 
Coverage: 0.96 
Length: 0.063 
End of Result 
------------- 
Result for Configuration 1 with Number of Users 900 
---------------------------------------------------- 
Mean Test Error: 0.03 
SD Test Error: 0.004 
Mean ECE: 0.017 
SD ECE: 0.004 
Coverage: 0.97 
Length: 0.055 
End of Result 
------------- 
Result for Configuration 1 with Number of Users 1200 
---------------------------------------------------- 
Mean Test Error: 0.026 
SD Test Error: 0.003 
Mean ECE: 0.015 
SD ECE: 0.004 
Coverage: 0.96 
Length: 0.051 
End of Result 
------------- 
Result for Config